# Colour Brings People Together
### Part C · Creating an empirically grounded colour palette

In Part B you audited your test palette: what you call each colour in your language, and what your audience calls it in theirs. Now reverse the direction, and fix the palette for each audience. You will give those **names**, yours and your audience's, to an AI system, ask it for the colour each name stands for, and check its answers against where speakers actually centre each name. Where the AI's colour diverges, you replace it with the Colour Spinner's best example, and you finish with a small palette you can defend with evidence. This matters: in the *Colour in Translation* benchmark (Mylonas et al., CHI 2026), a current large language model's colours were on average more than 17 ΔE from speakers' focal colours, because it mapped colour names onto the same prototypes whatever the language.

| Step | You do | Time |
|---|---|---|
| 1 | Load your names from Part B | 5 min |
| 2 | Ask an AI system for the best example of each name | 10 min |
| 3 | Compare its colours with the empirical best examples | 10 min |
| 4 | Replace divergent colours and check the result still works for every viewer | 15 min |
| 5 | Export the palette and prepare your finding | 5 min |
| | Plenary discussion | 15 min |

Run each cell with **Shift + Enter** and change only the grey form fields.

In [ ]:
#@title Run me first: load the masterclass toolkit
#@markdown The colour maths, the Colour Spinner connection and the charts live in `colour_masterclass.py`, published next to these notebooks.
ToolkitURL = "https://raw.githubusercontent.com/akvilesinkeviciute/colour-brings-people-together/v1.0/colour_masterclass.py"  #@param {type:"string"}
import os, importlib, urllib.request
try:                                   # always fetch the current version
    urllib.request.urlretrieve(ToolkitURL, "colour_masterclass.py")
except Exception as e:
    if not os.path.exists("colour_masterclass.py"):
        raise RuntimeError("Could not download the toolkit. Check your internet connection, or upload "
                           "colour_masterclass.py in Colab's Files panel (folder icon, left).") from e
    print("Couldn't refresh the toolkit, so using the copy already here.")
import colour_masterclass
importlib.reload(colour_masterclass)
from colour_masterclass import *

## 1 · Load your names from Part B

Upload the `partB_results.csv` you downloaded at the end of Part B. If you lost it, choose *Type them in* and list a few colour names per language instead; the notebook will look up their best examples.

In [ ]:
#@title Where your names come from
NamesFrom = "Upload partB_results.csv"  #@param ["Upload partB_results.csv", "Type them in"]
Backend = "Colour Spinner API"  #@param ["Colour Spinner API", "Copied from the browser"]
#@markdown *Copied from the browser* only matters if you type names in: then also type each name's best example (the Mapper's top-left tile) into the boxes below.

In [ ]:
#@title Only if you chose "Type them in"
if NamesFrom == "Type them in":
    name_boxes()      # choose a language, type a name; add its best example only for "Copied from the browser"
else:
    show("You're loading names from Part B, so there's nothing to fill in here.")

In [ ]:
names = load_names(NamesFrom)
LANGUAGES = list(dict.fromkeys(names.language))
show_names(names)

## 2 · Ask the AI system for the best example of each name

The prompt asks for the colour that a native speaker would consider the **best example** of each name: the most typical instance, not the average and not a brand colour. As in Part B, use the AI you use day to day, and the same one as in Part B so the two parts can be compared. The next cell gives you the prompt with **Copy prompt** and buttons for ChatGPT, Claude and Gemini, and numbered boxes: paste the colour code your AI gives for 1 into box 1, and so on.

In [ ]:
lines = "\n".join(numbered_items(names, "colour"))
PROMPT_C = f"""You are helping with a cross-cultural colour naming study.
For each numbered colour name below, give the sRGB colour, as a hex code, that native speakers of that language would consider the best example of the name: its most typical instance.

{lines}

Reply with one line per number, in this format and nothing else:
1. #RRGGBB
2. …"""
show_prompt(PROMPT_C, PASTE, rows=names, kind="colour")

In [ ]:
pal = read_ai_colours(collected_answers(), names)       # the colour codes pasted into the boxes
show_ai_palette(pal)

## 3 · Compare with how speakers actually use each name

For every name, the left half of each swatch is the AI's colour and the right half is the Colour Spinner's best example: the name's focal colour, the sample where speakers agree most on that name (the Mapper's top-left tile). The badge gives the CIEDE2000 difference between them.

A reminder of scale: about 1 is just noticeable with the two side by side, about 5 is obvious, and above 10 the two are usually read as different colours.

In [ ]:
#@title Fill in the Spinner's best examples and compare { run: "auto" }
ReplaceAbove = 10  #@param {type:"slider", min:3, max:30, step:1}
#@markdown An AI colour further than this CIEDE2000 colour difference from the speakers' best example counts as divergent.

pal = add_best_examples(pal, connect(Backend))     # looks up only the names that need it
pal["de00"] = [de00(ai, best) if isinstance(ai, str) and isinstance(best, str) else np.nan
               for ai, best in zip(pal.ai_hex, pal.spinner_hex)]
pal = pal.sort_values(["language", "de00"], ascending=[True, False]).reset_index(drop=True)
show_pairs(pal, threshold=ReplaceAbove)

The map below is the same top-down view of CAM16-UCS colour space used in Part B. Each arrow starts at the AI's colour and ends at the speakers' best example. Long arrows are large errors; arrows that all point the same way for one language suggest a systematic bias, for instance towards English focal colours.

In [ ]:
plot_ai_vs_speakers(pal, LANGUAGES, ReplaceAbove)
error_summary(pal, LANGUAGES, ReplaceAbove)

### Would speakers call the AI's colour by that name?

The distance above compares the AI's colour with a single point, the name's best example. But a name covers a range of colours, and the Spinner knows how speakers use it across that range. So send each AI colour back to the Spinner's Namer and ask what speakers would call it. If the answer is the name you gave the AI, the colour communicates as intended, even if it isn't the most typical example. If it's a different name, speakers would read the AI's colour as something else.

In [ ]:
spinner = connect(Backend)
pal["rt_name"], pal["rt_conf"] = None, np.nan
for k, row in pal.iterrows():
    if isinstance(row.ai_hex, str):
        namer = safe(spinner.name, row.ai_hex, row.language)   # what speakers call the AI's colour
        if namer:
            pal.at[k, "rt_name"], pal.at[k, "rt_conf"] = namer["name"], namer["confidence"]
show_round_trip(pal)

## 4 · Ground the palette

An AI colour diverges from observed language use if it is too far from the name's best example, or if speakers would call it by a different name. Choose which rule to apply in the form; by default a colour is replaced if either is true. A replaced colour becomes the Colour Spinner's best example of the name. The distance threshold is the slider in section 3.

In [ ]:
#@title Ground the palette { run: "auto" }
ReplaceWhen = "Either"  #@param ["Either", "Too far from the best example", "Speakers would use a different name"]

too_far = [pd.isna(d) or d > ReplaceAbove for d in pal.de00]
renamed = [isinstance(n, str) and not same_name(n, m, r)
           for n, m, r in zip(pal.rt_name, pal.name, pal.get("romanized", pd.Series(None, index=pal.index)))]
rule = {"Either": [a or b for a, b in zip(too_far, renamed)],
        "Too far from the best example": too_far,
        "Speakers would use a different name": renamed}[ReplaceWhen]

# Replace the AI's colour with the speakers' best example wherever the rule says it diverges.
pal["grounded_hex"] = [best if isinstance(best, str) and replace else ai
                       for ai, best, replace in zip(pal.ai_hex, pal.spinner_hex, rule)]
pal["replaced"] = pal.grounded_hex != pal.ai_hex
show_grounded(pal, LANGUAGES)

### Fix colours that grounding made too similar

Grounding can put two colours of one palette close together: the focal colours of neighbouring names, such as two blues, may be hard to tell apart for some viewers. The next cell checks every palette under all seven viewing conditions. Where two colours are closer than the threshold, it swaps one of them for **another colour speakers still call by the same name**: the next tiles in the Spinner's Mapper, which run from the most to the least typical. It picks the swap that separates the palette most. If no colour of that name helps, the table says so; then consider a second cue such as a pattern, label or icon.

In [ ]:
#@title Fix perceptual problems { run: "auto" }
ConfusableBelow = 10  #@param {type:"slider", min:3, max:25, step:1}
#@markdown Two colours closer than this CIEDE2000 colour difference, under any viewing condition, count as too similar.

if "repaired_from" in pal:                 # rerunning: start again from the grounded colours
    pal["grounded_hex"] = pal.repaired_from.fillna(pal.grounded_hex)
fixed = repair_palettes(pal, connect(Backend), LANGUAGES, ConfusableBelow)
if fixed is not None:
    pal = fixed
    show_repairs(pal, ConfusableBelow)

### Does the grounded palette still work for every viewer?

Grounding fixes the cognitive axis, but best examples of neighbouring names can sit close together. Check the perceptual axis again: for each language's palette, the smallest ΔE00 between any two of its colours under each viewing condition.

**A note on the distances.** All colour differences here are CIEDE2000 (ΔE00). CIEDE2000 was derived from colour-difference judgements by observers with typical colour vision. Applied to simulated colours, it estimates how different those colours would look to a typical observer, which only approximates what people with a colour vision deficiency or an older eye can tell apart.

In [ ]:
check = closest_pairs(pal, LANGUAGES)      # smallest ΔE00 within each palette, per condition
plot_closest_pairs(check, ConfusableBelow)
show_grounded_conditions(pal, LANGUAGES)

If a pair falls below the threshold under some condition, you have found a real design tension: the colours speakers expect for two names may be too similar for some viewers. Options to discuss: keep one colour and pick the next-best example of the other from the Spinner's Mapper panel, add a second cue such as a pattern, icon or label, or drop one name from the palette.

## 5 · Export your palette

The download has a picture of every palette, swatch files for design tools (Adobe, GIMP, Figma), CSS for websites, and the data with the evidence for every change.

### See the palette in use

Choose a language to see its grounded palette applied to three kinds of design: a **chart**, an **interface** (buttons and status dots) and **wayfinding** (colour-coded routes with labels). The second row shows the same designs for another kind of vision, so you can check the palette still works there. Each preview is also saved as a picture you can download.

In [ ]:
#@title Preview { run: "auto" }
PreviewLanguage = "British English"  #@param ["American English", "British English", "French", "Greek", "OtjiHimba", "Spanish"]
PreviewVision = "Deuteranopia"  #@param ["Typical", "Protanopia", "Deuteranopia", "Tritanopia", "Age 40", "Age 60", "Age 80"]

preview_applications(pal, PreviewLanguage, PreviewVision, path=f"preview_{PreviewLanguage.replace(' ', '_')}.png")

In [ ]:
export_palette(pal, LANGUAGES)

## 6 · Your most striking finding

The cell below picks the name where the AI system's colour was furthest from what speakers mean. You can choose a different one by typing its name in the form. Bring this card to the plenary.

In [ ]:
#@title Finding to share { run: "auto" }
FindingName = ""  #@param {type:"string"}
show_finding(pal, FindingName)

---
*Colour Spinner: Sinkeviciute, Ahmed, Stutters, Koliousis & Mylonas (2026), under review. Spin Colour Forest and the AI colour-naming benchmark: Mylonas, Ahmed, Sinkeviciute & Koliousis, CHI 2026. Colour vision deficiency and age simulations from the Colour Spinner. Colour science with the colour-science library; CAM16-UCS after Li et al. (2017).*